# 3.2b — ViT-B/32, fine-tuned

**One arm, one notebook.** Split out of `32_bigger_pretrained.ipynb`, which was interrupted.

State of that series:

| arm | | result |
|---|---|---|
| `15_resnet34_finetune` | 21.4M, 128px | **0.9041** — best in the project, and not truncated (best at 36 of 46) |
| `16_resnet50_finetune` | 24.0M, 128px | crashed at epoch 15, `Scratch` still climbing (0.741 → 0.765) |
| `17_vit_b_32_finetune` | 87.7M, 224px | **never ran — this notebook** |

## What this arm asks

The from-scratch models top out near 0.89–0.90 with **2.8M** parameters. This is **87.7M**,
pretrained on ImageNet, with attention instead of convolution. If it lands in the same
place, the ceiling is the data and not the model — which is what capacity said within
ConvNeXt (414k → 2.68M was −0.002) and what ResNet34-vs-ResNet18 said on the pretrained
side (+0.010, inside the noise floor).

## Two things it is NOT compared fairly on

**224x224 is forced.** Torchvision ViT checkpoints carry positional embeddings fitted to
exactly that size and raise on anything else, so this arm cannot share the 128px geometry
of every other v32/v33 arm. The input is a further upsample of a 212x187 map, and 224²
exceeds the 2 GiB geometry-cache budget so the loader redoes letterboxing per access.

**32x32 patches.** Over 224px that is a 7x7 grid. A one-die-wide `Scratch` is exactly the
pattern a 32-pixel patch swallows whole — the same objection as downsampling, which is the
thread running through this entire project. ViT-B/16 would give a 14x14 grid and *fewer*
parameters (86.0M), but ~3x the compute: 1482 s/epoch against 458 on a T4.

Both belong in the presentation rather than a footnote. Roughly **3.5–4 h on an L4**.

## To run ResNet50 instead

Change `ARM` at the top of section 3 to `"16_resnet50_finetune"`. Its Drive checkpoint
means it resumes from epoch 15 rather than restarting, so it is the cheaper of the two.

## Before you start

1. `git pull` in `/content/fdl-project`, then **Runtime > Restart session** — Colab caches
   `fdl_project` after the first import.
2. `wandb login`, or `WANDB_KEY` in Colab Secrets.
3. Run All.

## 0. Colab web UI only — clone and authenticate

Skip if `/content/fdl-project` already exists.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


## 3. Train

`ARM` at the top selects which config runs — the ViT by default, ResNet50 if you switch it.

The cell prints the parameter groups before training: the encoder should be the large
number at lr 1e-05 and the head the small one at 0.0007. It also checks the input size
against what the backbone requires, so a mismatch fails in a second rather than after the
2 GB pickle has loaded.

In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.training.optim import build_optimizer
from fdl_project.training.runner import run_experiment

SERIES = "v32_bigger_pretrained"

# "17_vit_b_32_finetune"  the arm that never ran
# "16_resnet50_finetune"  crashed at epoch 15; resumes from its Drive checkpoint
ARM = "17_vit_b_32_finetune"

CONFIG = REPO / "configs/train" / SERIES / f"{ARM}.yaml"
assert CONFIG.exists(), f"{CONFIG} missing -- git pull, then Runtime > Restart session"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},finetune]"]

config = load_experiment_config(CONFIG, overrides=OVERRIDES)
assert config.model.kwargs["freeze_encoder"] is False, "this arm fine-tunes"
assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"

model = build_model(config.model.name, **config.model.kwargs)
required = model.required_input_size
size = tuple(config.data.preprocessing.target_size)
assert required is None or size == (required, required), (
    f"{config.model.kwargs['architecture']} requires {required}px, config says {size}"
)
optimizer = build_optimizer(model, config.optimizer)
print(f"=== {config.name}  ({config.model.kwargs['architecture']}, {size[0]}px)")
for group in optimizer.param_groups:
    print(f"    {str(group.get('name')):10} lr={group['lr']:<9g} "
          f"{sum(p.numel() for p in group['params']):>12,} parameters")
print(f"    {sum(p.numel() for p in model.parameters()):,} total, "
      f"max_epochs {config.trainer.max_epochs}, patience "
      f"{config.trainer.early_stopping.patience}")
del model, optimizer

dataframe = load_wm811k_dataframe(DATASET)
started = time.monotonic()
result = run_experiment(config, overwrite=True, dataframe=dataframe)

macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
per_class = result.validation.per_class_metrics.set_index("class_name")["f1"]
row = {
    "run": config.name,
    "backbone": config.model.kwargs["architecture"],
    "px": size[0],
    "macro_f1": round(float(macro.point_estimate), 4),
    "ci_lower": round(float(macro.ci_lower), 4),
    "ci_upper": round(float(macro.ci_upper), 4),
    "scratch_f1": round(float(per_class["Scratch"]), 3),
    "near_full_f1": round(float(per_class["Near-full"]), 3),
    "best_epoch": result.fit.best_epoch,
    "epochs": len(result.fit.history),
    "minutes": round((time.monotonic() - started) / 60, 1),
}
csv = OUTPUT / f"{ARM}_results.csv"
pd.DataFrame([row]).to_csv(csv, index=False)
if HAS_DRIVE:
    shutil.copy2(csv, DRIVE / f"{SERIES}_{ARM}.csv")

print(f"\n  macro-F1 {row['macro_f1']:.4f} [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")
print(f"  Scratch {row['scratch_f1']:.3f}   Near-full {row['near_full_f1']:.3f}")
print(f"  best epoch {row['best_epoch']}/{row['epochs']}   {row['minutes']:.1f} min")
if row["best_epoch"] >= row["epochs"] - 2:
    print("  STILL IMPROVING AT THE CAP -- this number is a floor.")
print(f"  saved to {csv}")

## 4. Where it lands

Run after section 3 finishes.

In [ ]:
REFERENCE = [
    ("v32-resnet34_finetune",      "pretrained 21.4M, 128px", 0.9041, 0.793),
    ("dilated-style-64-dihedral8", "ours 298k, old aug",      0.8995, 0.793),
    ("v28-convnext_big_128",       "ours 2.68M, 128px",       0.8988, 0.787),
    ("v30-finetune_encoder_1e-5",  "pretrained 11.3M, 128px", 0.8938, 0.813),
    ("v27-resnet_style",           "ours 2.83M, 64px",        0.8900, 0.759),
    ("v27-convnext_style",         "ours 414k, 64px",         0.8883, 0.736),
    ("v27-baseline_cnn",           "ours 157k, 64px",         0.8646, 0.715),
    ("v29-resnet18_frozen_onehot", "pretrained FROZEN",       0.7351, 0.401),
]
NOISE_FLOOR = 0.02

table = pd.DataFrame(REFERENCE, columns=["run", "what", "macro_f1", "scratch_f1"])
mine = pd.DataFrame([{"run": row["run"], "what": "THIS NOTEBOOK",
                      "macro_f1": row["macro_f1"], "scratch_f1": row["scratch_f1"]}])
pd.set_option("display.width", 210)
display(pd.concat([mine, table]).sort_values("macro_f1", ascending=False).reset_index(drop=True))

for label, reference in [("v32-resnet34_finetune, the best so far", 0.9041),
                         ("v30-finetune_encoder_1e-5 (resnet18)",   0.8938),
                         ("v27-convnext_style, ours at 414k",       0.8883)]:
    delta = row["macro_f1"] - reference
    verdict = "REAL" if abs(delta) > NOISE_FLOOR else "inside the noise floor"
    print(f"vs {label:40} {delta:+.4f}  [{verdict}]")

## 5. Report back

```
vit_b_32: macro-F1 X.XXXX [lo, hi], Scratch X.XXX, best epoch N/M, T min
```

Flag it if it says **STILL IMPROVING AT THE CAP**.

CSV at `output/v32_bigger_pretrained/17_vit_b_32_finetune_results.csv`, copied to Drive.